# OJP Pilot – bereinigte Version

Dieses Notebook ist so aufgebaut, dass es **von oben nach unten** ausgeführt werden kann.

1. Imports, API-Token und URL laden
2. Funktion zur Datensammlung definieren
3. Funktion zur Stationssuche definieren
4. Zürich HB testen
5. Bern suchen
6. Bern Bahn sowie Bern Tram/Bus testen

Der API-Token wird nur über `getpass()` eingegeben und nicht im Notebook gespeichert.

In [10]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from uuid import uuid4
from xml.sax.saxutils import escape

URL = "https://api.opentransportdata.swiss/ojp20"

TOKEN = getpass("OJP API Token: ").strip()

if not TOKEN:
    raise ValueError("Der OJP API Token ist leer.")


## 1. OJP-Daten für eine Haltestelle sammeln

In [11]:
def collect_ojp_station(
    stop_id,
    stop_name,
    token,
    number_of_results=30
):
    """
    Ruft OJP-Echtzeitdaten für eine Haltestelle ab,
    parst die XML-Antwort und gibt einen pandas DataFrame zurück.
    """

    # 1. Zeitpunkt des API-Abrufs
    now_utc = datetime.now(timezone.utc)
    timestamp = (
        now_utc
        .isoformat(timespec="milliseconds")
        .replace("+00:00", "Z")
    )
    message_id = f"zhaw-project-{uuid4()}"
    stop_name_xml = escape(stop_name)

    # 2. XML Request
    xml_request = f"""<?xml version="1.0" encoding="UTF-8"?>
<OJP
    xmlns="http://www.vdv.de/ojp"
    xmlns:siri="http://www.siri.org.uk/siri"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema"
    xsi:schemaLocation="http://www.vdv.de/ojp"
    version="2.0">

    <OJPRequest>
        <siri:ServiceRequest>

            <siri:ServiceRequestContext>
                <siri:Language>de</siri:Language>
            </siri:ServiceRequestContext>

            <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>
            <siri:RequestorRef>ZHAW_DataAnalytics_Project</siri:RequestorRef>

            <OJPStopEventRequest>

                <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>
                <siri:MessageIdentifier>{message_id}</siri:MessageIdentifier>

                <Location>
                    <PlaceRef>
                        <siri:StopPointRef>{stop_id}</siri:StopPointRef>
                        <Name>
                            <Text>{stop_name_xml}</Text>
                        </Name>
                    </PlaceRef>
                    <DepArrTime>{timestamp}</DepArrTime>
                </Location>

                <Params>
                    <NumberOfResults>{number_of_results}</NumberOfResults>
                    <StopEventType>departure</StopEventType>
                    <IncludePreviousCalls>false</IncludePreviousCalls>
                    <IncludeOnwardCalls>false</IncludeOnwardCalls>
                    <UseRealtimeData>full</UseRealtimeData>
                </Params>

            </OJPStopEventRequest>

        </siri:ServiceRequest>
    </OJPRequest>

</OJP>
"""

    # 3. API Request
    headers = {
        "Content-Type": "application/xml",
        "Authorization": f"Bearer {token}"
    }

    response = requests.post(
        URL,
        headers=headers,
        data=xml_request.encode("utf-8"),
        timeout=30
    )

    print(f"{stop_name}: HTTP {response.status_code}")

    if response.status_code != 200:
        print("\nAPI-Fehler:")
        print(response.text[:1000])

    response.raise_for_status()

    # 4. Raw XML speichern
    safe_station_name = (
        stop_name
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace(",", "")
        .replace("ü", "ue")
        .replace("ö", "oe")
        .replace("ä", "ae")
        .replace("é", "e")
        .replace("è", "e")
    )

    raw_dir = Path("data/raw/ojp")
    raw_dir.mkdir(parents=True, exist_ok=True)

    raw_filename = (
        raw_dir
        / f"{safe_station_name}_{now_utc.strftime('%Y%m%d_%H%M%S')}.xml"
    )
    raw_filename.write_bytes(response.content)

    # 5. XML parsen
    root = ET.fromstring(response.content)

    ns = {
        "ojp": "http://www.vdv.de/ojp",
        "siri": "http://www.siri.org.uk/siri"
    }

    results = root.findall(".//ojp:StopEventResult", ns)
    print(f"{stop_name}: {len(results)} Verbindungen")

    # 6. Daten extrahieren
    rows = []

    for result in results:
        stop_event = result.find("ojp:StopEvent", ns)

        if stop_event is None:
            continue

        this_call = stop_event.find(
            "ojp:ThisCall/ojp:CallAtStop",
            ns
        )
        service = stop_event.find(
            "ojp:Service",
            ns
        )

        if this_call is None or service is None:
            continue

        rows.append({
            "collection_timestamp": timestamp,
            "station_id": stop_id,
            "station_name": stop_name,
            "stop_point_ref": this_call.findtext(
                "siri:StopPointRef",
                default=None,
                namespaces=ns
            ),
            "operating_day": service.findtext(
                "ojp:OperatingDayRef",
                default=None,
                namespaces=ns
            ),
            "journey_ref": service.findtext(
                "ojp:JourneyRef",
                default=None,
                namespaces=ns
            ),
            "transport_mode": service.findtext(
                "ojp:Mode/ojp:PtMode",
                default=None,
                namespaces=ns
            ),
            "product_category": service.findtext(
                "ojp:ProductCategory/ojp:Name/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "public_code": service.findtext(
                "ojp:PublicCode",
                default=None,
                namespaces=ns
            ),
            "line": service.findtext(
                "ojp:PublishedServiceName/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "train_number": service.findtext(
                "ojp:TrainNumber",
                default=None,
                namespaces=ns
            ),
            "origin": service.findtext(
                "ojp:OriginText/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "destination": service.findtext(
                "ojp:DestinationText/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "planned_platform": this_call.findtext(
                "ojp:PlannedQuay/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "estimated_platform": this_call.findtext(
                "ojp:EstimatedQuay/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "scheduled_departure": this_call.findtext(
                "ojp:ServiceDeparture/ojp:TimetabledTime",
                default=None,
                namespaces=ns
            ),
            "estimated_departure": this_call.findtext(
                "ojp:ServiceDeparture/ojp:EstimatedTime",
                default=None,
                namespaces=ns
            )
        })

    # 7. DataFrame erstellen
    df = pd.DataFrame(rows)

    if df.empty:
        print(f"Keine Daten für {stop_name} gefunden.")
        return df

    # 8. Zeitvariablen
    time_columns = [
        "collection_timestamp",
        "scheduled_departure",
        "estimated_departure"
    ]

    for column in time_columns:
        df[column] = pd.to_datetime(
            df[column],
            utc=True,
            errors="coerce"
        )

    # 9. Realtime und Delay
    df["has_realtime"] = df["estimated_departure"].notna()

    # Fehlende EstimatedTime bleibt NaN und wird nicht als 0 Minuten interpretiert.
    df["predicted_delay_minutes"] = (
        df["estimated_departure"]
        - df["scheduled_departure"]
    ).dt.total_seconds() / 60

    # 10. Schweizer Lokalzeit
    df["collection_timestamp_local"] = (
        df["collection_timestamp"].dt.tz_convert("Europe/Zurich")
    )
    df["scheduled_departure_local"] = (
        df["scheduled_departure"].dt.tz_convert("Europe/Zurich")
    )
    df["estimated_departure_local"] = (
        df["estimated_departure"].dt.tz_convert("Europe/Zurich")
    )

    # 11. Zeitliche Merkmale
    df["date"] = df["scheduled_departure_local"].dt.date
    df["hour"] = df["scheduled_departure_local"].dt.hour
    df["weekday"] = df["scheduled_departure_local"].dt.day_name()
    df["weekend"] = df["scheduled_departure_local"].dt.dayofweek >= 5
    df["minutes_until_departure"] = (
        df["scheduled_departure"]
        - df["collection_timestamp"]
    ).dt.total_seconds() / 60

    # 12. CSV Snapshot speichern
    interim_dir = Path("data/interim")
    interim_dir.mkdir(parents=True, exist_ok=True)

    csv_filename = (
        interim_dir
        / f"{safe_station_name}_{now_utc.strftime('%Y%m%d_%H%M%S')}.csv"
    )

    df.to_csv(csv_filename, index=False)
    print(f"Gespeichert: {csv_filename}")

    return df


## 2. Haltestellen über ihren Namen suchen

In [3]:
def find_ojp_stations(
    search_name,
    token,
    number_of_results=5
):
    """
    Sucht OJP-Haltestellen anhand eines Namens
    und gibt die gefundenen Stop-IDs als DataFrame zurück.
    """

    # 1. Zeitpunkt und Message-ID
    now_utc = datetime.now(timezone.utc)
    timestamp = (
        now_utc
        .isoformat(timespec="milliseconds")
        .replace("+00:00", "Z")
    )
    message_id = f"zhaw-location-{uuid4()}"
    search_name_xml = escape(search_name)

    # 2. OJP LocationInformationRequest
    xml_request = f"""<?xml version="1.0" encoding="UTF-8"?>
<OJP
    xmlns="http://www.vdv.de/ojp"
    xmlns:siri="http://www.siri.org.uk/siri"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    version="2.0">

    <OJPRequest>
        <siri:ServiceRequest>

            <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>
            <siri:RequestorRef>ZHAW_DataAnalytics_Project</siri:RequestorRef>

            <OJPLocationInformationRequest>

                <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>
                <siri:MessageIdentifier>{message_id}</siri:MessageIdentifier>

                <InitialInput>
                    <Name>{search_name_xml}</Name>
                </InitialInput>

                <Restrictions>
                    <Type>stop</Type>
                    <NumberOfResults>{number_of_results}</NumberOfResults>
                    <IncludePtModes>true</IncludePtModes>
                </Restrictions>

            </OJPLocationInformationRequest>

        </siri:ServiceRequest>
    </OJPRequest>

</OJP>
"""

    # 3. API Request
    headers = {
        "Content-Type": "application/xml",
        "Authorization": f"Bearer {token}"
    }

    response = requests.post(
        URL,
        headers=headers,
        data=xml_request.encode("utf-8"),
        timeout=30
    )

    print(
        f"Stationssuche '{search_name}': "
        f"HTTP {response.status_code}"
    )

    if response.status_code != 200:
        print("\nAPI-Fehler:")
        print(response.text[:1000])

    response.raise_for_status()

    # 4. XML parsen
    root = ET.fromstring(response.content)

    ns = {
        "ojp": "http://www.vdv.de/ojp",
        "siri": "http://www.siri.org.uk/siri"
    }

    place_results = root.findall(
        ".//ojp:PlaceResult",
        ns
    )

    # 5. Treffer extrahieren
    rows = []

    for result in place_results:
        place = result.find("ojp:Place", ns)

        if place is None:
            continue

        stop_place = place.find("ojp:StopPlace", ns)

        if stop_place is None:
            continue

        modes = []

        for mode in place.findall("ojp:Mode", ns):
            pt_mode = mode.findtext(
                "ojp:PtMode",
                default=None,
                namespaces=ns
            )

            if pt_mode is not None:
                modes.append(pt_mode)

        rows.append({
            "stop_id": stop_place.findtext(
                "ojp:StopPlaceRef",
                default=None,
                namespaces=ns
            ),
            "stop_name": stop_place.findtext(
                "ojp:StopPlaceName/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "modes": ", ".join(modes),
            "latitude": place.findtext(
                "ojp:GeoPosition/siri:Latitude",
                default=None,
                namespaces=ns
            ),
            "longitude": place.findtext(
                "ojp:GeoPosition/siri:Longitude",
                default=None,
                namespaces=ns
            ),
            "probability": result.findtext(
                "ojp:Probability",
                default=None,
                namespaces=ns
            ),
            "complete": result.findtext(
                "ojp:Complete",
                default=None,
                namespaces=ns
            )
        })

    # 6. DataFrame
    df_stations = pd.DataFrame(rows)

    if not df_stations.empty:
        df_stations["latitude"] = pd.to_numeric(
            df_stations["latitude"],
            errors="coerce"
        )
        df_stations["longitude"] = pd.to_numeric(
            df_stations["longitude"],
            errors="coerce"
        )
        df_stations["probability"] = pd.to_numeric(
            df_stations["probability"],
            errors="coerce"
        )

    print("Gefundene Haltestellen:", len(df_stations))

    return df_stations


## 3. Funktionstest: Zürich HB

In [4]:
df_zurich = collect_ojp_station(
    stop_id="ch:1:sloid:3000",
    stop_name="Zürich HB",
    token=TOKEN,
    number_of_results=30
)

display(
    df_zurich[
        [
            "station_name",
            "transport_mode",
            "product_category",
            "line",
            "destination",
            "scheduled_departure_local",
            "estimated_departure_local",
            "predicted_delay_minutes"
        ]
    ].head(10)
)


Zürich HB: HTTP 200
Zürich HB: 30 Verbindungen
Gespeichert: data/interim/zuerich_hb_20260922_112018.csv


,station_name,transport_mode,product_category,line,destination,scheduled_departure_local,estimated_departure_local,predicted_delay_minutes
0,Zürich HB,rail,S-Bahn,S14,Affoltern am Albis,2026-09-22 13:19:00+02:00,2026-09-22 13:20:00+02:00,1.0
1,Zürich HB,rail,S-Bahn,S24,Zug,2026-09-22 13:21:00+02:00,2026-09-22 13:21:18+02:00,0.3
2,Zürich HB,rail,S-Bahn,S15,Niederweningen,2026-09-22 13:22:00+02:00,2026-09-22 13:22:24+02:00,0.4
3,Zürich HB,rail,S-Bahn,S5,Pfäffikon SZ,2026-09-22 13:24:00+02:00,2026-09-22 13:24:18+02:00,0.3
4,Zürich HB,rail,S-Bahn,S8,Winterthur,2026-09-22 13:25:00+02:00,2026-09-22 13:25:18+02:00,0.3
5,Zürich HB,rail,S-Bahn,S3,Zürich Hardbrücke,2026-09-22 13:26:00+02:00,2026-09-22 13:27:18+02:00,1.3
6,Zürich HB,rail,S-Bahn,S9,Uster,2026-09-22 13:28:00+02:00,2026-09-22 13:28:18+02:00,0.3
7,Zürich HB,rail,InterRegio,IR55,Biel/Bienne,2026-09-22 13:29:00+02:00,2026-09-22 13:29:24+02:00,0.4
8,Zürich HB,rail,S-Bahn,S11,Aarau,2026-09-22 13:29:00+02:00,2026-09-22 13:30:06+02:00,1.1
9,Zürich HB,rail,S-Bahn,S6,Uetikon,2026-09-22 13:30:00+02:00,2026-09-22 13:30:30+02:00,0.5


## 4. Stationssuche: Bern

In [5]:
df_station_search = find_ojp_stations(
    search_name="Bern",
    token=TOKEN,
    number_of_results=5
)

display(df_station_search)


Stationssuche 'Bern': HTTP 200
Gefundene Haltestellen: 6


,stop_id,stop_name,modes,latitude,longitude,probability,complete
0,ch:1:sloid:7000,Bern,"rail, rail, rail, rail",46.94883,7.43913,1.000,true
1,ch:1:sloid:88699,"Bern, Wankdorf Center","tram, bus",46.96165,7.46590,0.865,true
2,ch:1:sloid:76646,"Bern, Bahnhof","tram, bus",46.94811,7.44021,0.853,true
3,ch:1:sloid:7062,Muri b. Bern,"tram, bus",46.93141,7.48641,0.851,true
4,ch:1:sloid:88174,"Bern, Weltpostverein","tram, bus",46.93882,7.47205,0.844,true
5,ch:1:sloid:4106,Bern Bümpliz Süd,rail,46.93749,7.39524,0.817,true


## 5. Bern: Bahn sowie Tram/Bus abrufen

In [6]:
df_bern_rail = collect_ojp_station(
    stop_id="ch:1:sloid:7000",
    stop_name="Bern",
    token=TOKEN,
    number_of_results=30
)

df_bern_local = collect_ojp_station(
    stop_id="ch:1:sloid:76646",
    stop_name="Bern, Bahnhof",
    token=TOKEN,
    number_of_results=30
)

print("\nBern Bahn:")
display(df_bern_rail["transport_mode"].value_counts(dropna=False))

print("\nBern Bahnhof Tram/Bus:")
display(df_bern_local["transport_mode"].value_counts(dropna=False))

display(
    df_bern_local[
        [
            "station_name",
            "transport_mode",
            "product_category",
            "line",
            "destination",
            "scheduled_departure_local",
            "estimated_departure_local",
            "predicted_delay_minutes"
        ]
    ].head(15)
)


Bern: HTTP 200
Bern: 30 Verbindungen
Gespeichert: data/interim/bern_20260922_112025.csv
Bern, Bahnhof: HTTP 200
Bern, Bahnhof: 30 Verbindungen
Gespeichert: data/interim/bern_bahnhof_20260922_112025.csv

Bern Bahn:


transport_mode
rail    30
Name: count, dtype: int64


Bern Bahnhof Tram/Bus:


transport_mode
bus     22
tram     8
Name: count, dtype: int64

,station_name,transport_mode,product_category,line,destination,scheduled_departure_local,estimated_departure_local,predicted_delay_minutes
0,"Bern, Bahnhof",bus,Bus,10,Ostermundigen Rüti,2026-09-22 13:21:00+02:00,2026-09-22 13:21:00+02:00,0.0
1,"Bern, Bahnhof",bus,Bus,19,Elfenau,2026-09-22 13:21:00+02:00,2026-09-22 13:21:30+02:00,0.5
2,"Bern, Bahnhof",bus,Bus,7A,Ostring,2026-09-22 13:21:00+02:00,2026-09-22 13:21:00+02:00,0.0
3,"Bern, Bahnhof",tram,Tram,9,Wabern,2026-09-22 13:21:00+02:00,2026-09-22 13:21:00+02:00,0.0
4,"Bern, Bahnhof",bus,Bus,12,Zentrum Paul Klee,2026-09-22 13:21:00+02:00,2026-09-22 13:21:12+02:00,0.2
5,"Bern, Bahnhof",bus,Bus,20,Wankdorf Bhf,2026-09-22 13:21:48+02:00,2026-09-22 13:21:48+02:00,0.0
6,"Bern, Bahnhof",bus,Bus,8A,Saali,2026-09-22 13:22:48+02:00,2026-09-22 13:22:48+02:00,0.0
7,"Bern, Bahnhof",tram,Tram,7,Bümpliz,2026-09-22 13:23:12+02:00,2026-09-22 13:23:12+02:00,0.0
8,"Bern, Bahnhof",tram,Tram,3,Weissenbühl,2026-09-22 13:24:00+02:00,2026-09-22 13:24:00+02:00,0.0
9,"Bern, Bahnhof",bus,Bus,12,Holligen,2026-09-22 13:24:00+02:00,2026-09-22 13:23:36+02:00,-0.4


In [12]:
cities = [
    "Zürich",
    "Bern",
    "Basel",
    "Luzern",
    "St. Gallen",
    "Lausanne",
    "Genève",
    "Lugano"
]

station_search_results = {}

for city in cities:
    print("\n" + "=" * 60)
    print(city)
    print("=" * 60)

    result = find_ojp_stations(
        search_name=city,
        token=TOKEN,
        number_of_results=8
    )

    station_search_results[city] = result

    display(
        result[
            [
                "stop_id",
                "stop_name",
                "modes",
                "probability"
            ]
        ]
    )


Zürich
Stationssuche 'Zürich': HTTP 200
Gefundene Haltestellen: 9


,stop_id,stop_name,modes,probability
0,ch:1:sloid:3000,Zürich HB,"rail, rail, rail, rail",0.973
1,ch:1:sloid:3003,Zürich Stadelhofen,rail,0.878
2,ch:1:sloid:3004,Zürich Tiefenbrunnen,rail,0.876
3,ch:1:sloid:91365,"Zürich, Sihlcity",bus,0.858
4,ch:1:sloid:91446,"Zürich, Zwielplatz","tram, bus",0.855
5,ch:1:sloid:91366,"Zürich, Sihlcity Nord","tram, bus",0.848
6,ch:1:sloid:3001,Zürich Altstetten,"rail, rail",0.830
7,ch:1:sloid:3054,Zürich Triemli,rail,0.797
8,ch:1:sloid:3610,"Zürich, Triemliplatz","tram, bus",0.789



Bern
Stationssuche 'Bern': HTTP 200
Gefundene Haltestellen: 9


,stop_id,stop_name,modes,probability
0,ch:1:sloid:7000,Bern,"rail, rail, rail, rail",1.000
1,ch:1:sloid:88699,"Bern, Wankdorf Center","tram, bus",0.865
2,ch:1:sloid:76646,"Bern, Bahnhof","tram, bus",0.853
3,ch:1:sloid:7062,Muri b. Bern,"tram, bus",0.851
4,ch:1:sloid:88174,"Bern, Weltpostverein","tram, bus",0.844
5,ch:1:sloid:4106,Bern Bümpliz Süd,rail,0.817
6,ch:1:sloid:89226,"Bern Bümpliz Süd, Bahnhof",bus,0.811
7,ch:1:sloid:4108,Bern Europaplatz,"rail, rail",0.784
8,ch:1:sloid:80939,"Bern Europaplatz, Bahnhof","tram, bus",0.776



Basel
Stationssuche 'Basel': HTTP 200
Gefundene Haltestellen: 10


...

,stop_id,stop_name,modes,probability
0,ch:1:sloid:1120,Lausanne,"rail, rail, rail, rail",0.900
1,ch:1:sloid:1181,Lausanne-Flon,rail,0.869
2,ch:1:sloid:92017,"Lausanne, CHUV","metro, bus",0.769
3,ch:1:sloid:92024,"Lausanne, Cour",bus,0.769
4,ch:1:sloid:91818,"Lausanne, Flon",metro,0.769
5,ch:1:sloid:92050,"Lausanne, gare","metro, bus",0.769
6,ch:1:sloid:92057,"Lausanne, Grey",bus,0.769
7,ch:1:sloid:79237,"Lausanne, Ours","metro, bus",0.769
8,ch:1:sloid:92082,"Lausanne, Motte",bus,0.766
9,ch:1:sloid:79245,"Lausanne, Foyer",bus,0.766



Genève
Stationssuche 'Genève': HTTP 200
Gefundene Haltestellen: 9


,stop_id,stop_name,modes,probability
0,ch:1:sloid:1008,Genève,"bus, rail, rail, rail, rail",0.900
1,ch:1:sloid:1237,Genève CGN,water,0.768
2,ch:1:sloid:92776,"Genève, Gos",bus,0.768
3,ch:1:sloid:92818,"Genève, Dôle",bus,0.765
4,ch:1:sloid:92835,"Genève, Guye",bus,0.765
5,ch:1:sloid:92850,"Genève, Lyon","tram, bus",0.765
6,ch:1:sloid:92863,"Genève, Môle","tram, bus",0.765
7,ch:1:sloid:92894,"Genève, Rieu",bus,0.765
8,ch:1:sloid:87061,"Genève, Rive","tram, bus",0.765



Lugano
Stationssuche 'Lugano': HTTP 200
Gefundene Haltestellen: 10


,stop_id,stop_name,modes,probability
0,ch:1:sloid:5300,Lugano,"rail, rail, rail, rail",1.000
1,ch:1:sloid:11123,"Lugano, Stazione/Via Basilea",bus,0.777
2,ch:1:sloid:5391,Lugano FLP,rail,0.768
3,ch:1:sloid:91603,"Lugano, Centro",bus,0.760
4,ch:1:sloid:91605,"Lugano, Loreto",bus,0.760
5,ch:1:sloid:91610,"Lugano, Resega",bus,0.760
6,ch:1:sloid:91618,"Lugano, Tassino",bus,0.758
7,ch:1:sloid:79006,"Lugano, Vignola",bus,0.758
8,ch:1:sloid:5395,Lugano Airport,rail,0.758
9,ch:1:sloid:75310,"Lugano, Genzana",bus,0.758


In [13]:
rail_stations = [
    {"city": "Zürich",     "stop_id": "ch:1:sloid:3000", "stop_name": "Zürich HB"},
    {"city": "Bern",       "stop_id": "ch:1:sloid:7000", "stop_name": "Bern"},
    {"city": "Basel",      "stop_id": "ch:1:sloid:10",   "stop_name": "Basel SBB"},
    {"city": "Luzern",     "stop_id": "ch:1:sloid:5000", "stop_name": "Luzern"},
    {"city": "St. Gallen", "stop_id": "ch:1:sloid:6302", "stop_name": "St. Gallen"},
    {"city": "Lausanne",   "stop_id": "ch:1:sloid:1120", "stop_name": "Lausanne"},
    {"city": "Genève",     "stop_id": "ch:1:sloid:1008", "stop_name": "Genève"},
    {"city": "Lugano",     "stop_id": "ch:1:sloid:5300", "stop_name": "Lugano"}
]

In [14]:
local_station_searches = [
    "Zürich, Bahnhof",
    "Basel, Bahnhof",
    "Luzern, Bahnhof",
    "St. Gallen, Bahnhof",
    "Lausanne, gare",
    "Genève, gare",
    "Lugano, Stazione"
]

local_search_results = {}

for search_name in local_station_searches:

    print("\n" + "=" * 60)
    print(search_name)
    print("=" * 60)

    result = find_ojp_stations(
        search_name=search_name,
        token=TOKEN,
        number_of_results=8
    )

    local_search_results[search_name] = result

    display(
        result[
            [
                "stop_id",
                "stop_name",
                "modes",
                "probability"
            ]
        ]
    )


Zürich, Bahnhof
Stationssuche 'Zürich, Bahnhof': HTTP 200
Gefundene Haltestellen: 9


,stop_id,stop_name,modes,probability
0,ch:1:sloid:2495,"Zürich Wollishofen, Bahnhof",bus,0.804
1,ch:1:sloid:91058,"Zürich Enge, Bahnhof","tram, bus",0.780
2,ch:1:sloid:91064,"Zürich Selnau, Bahnhof",tram,0.775
3,ch:1:sloid:91061,"Zürich Leimbach, Bahnhof",bus,0.772
4,ch:1:sloid:80449,"Zürich Oerlikon, Bahnhof","tram, bus",0.772
5,ch:1:sloid:73710,"Zürich Wiedikon, Bahnhof","tram, bus",0.772
6,ch:1:sloid:73205,"Zürich Flughafen, Bahnhof","tram, bus",0.770
7,ch:1:sloid:91054,"Zürich Affoltern, Bahnhof",bus,0.770
8,ch:1:sloid:91066,"Zürich Wipkingen, Bahnhof",bus,0.770



Basel, Bahnhof
Stationssuche 'Basel, Bahnhof': HTTP 200
Gefundene Haltestellen: 6


,stop_id,stop_name,modes,probability
0,ch:1:sloid:90,Basel Bad Bf,"bus, rail, rail, rail",0.780
1,ch:1:sloid:78143,"Basel, Bahnhof SBB","tram, bus",0.780
2,8014431,Basel Bad Bf,rail,0.780
3,ch:1:sloid:92321,"Basel, Badischer Bahnhof","tram, bus",0.768
4,ch:1:sloid:92322,"Basel, Bahnhof St. Johann","tram, bus",0.765
5,flx:RB,Basel Bad Bf (FlixTrain) - Bus Station,rail,0.748



Luzern, Bahnhof
Stationssuche 'Luzern, Bahnhof': HTTP 200
Gefundene Haltestellen: 5


...

,stop_id,stop_name,modes,probability
0,ch:1:sloid:92050,"Lausanne, gare","metro, bus",0.800
1,ch:1:sloid:7628,"Romanel-sur-Lausanne, gare",bus,0.762



Genève, gare
Stationssuche 'Genève, gare': HTTP 200
Gefundene Haltestellen: 9


,stop_id,stop_name,modes,probability
0,ch:1:sloid:92886,"Genève-Champel, gare",bus,0.769
1,ch:1:sloid:87057,"Genève, gare Cornavin","tram, bus",0.768
2,ch:1:sloid:92895,"Genève-Sécheron, gare",bus,0.768
3,ch:1:sloid:92829,"Genève-Eaux-Vives, gare","tram, bus",0.761
4,ch:1:sloid:92898,"Genève-Champel, gare/Hôpital",bus,0.756
5,ch:1:sloid:92935,"Genève-Aéroport, gare-Arena",bus,0.756
6,ch:1:sloid:92878,"Genève-Champel, gare/Peschier",bus,0.755
7,ch:1:sloid:2056,"Genève-Aéroport, gare routière",bus,0.754
8,ch:1:sloid:92865,"Genève-Eaux-Vives, gare/Bloch",bus,0.753



Lugano, Stazione
Stationssuche 'Lugano, Stazione': HTTP 200
Gefundene Haltestellen: 6


,stop_id,stop_name,modes,probability
0,ch:1:sloid:11123,"Lugano, Stazione/Via Basilea",bus,0.802
1,ch:1:sloid:5380,"Lugano, Stazione",bus,0.800
2,ch:1:sloid:91805,"Lugano, Stazione Nord",bus,0.780
3,ch:1:sloid:30462,Lugano Stazione (funicolare),telecabin,0.768
4,ch:1:sloid:75286,"Lugano, Stazione Piazza Besso",bus,0.764
5,ch:1:sloid:75273,"Lugano, Stazione Via Sorengo",bus,0.764
